# Job Offers Pipeline - Development Notebook

## Objetivo
Este notebook orquesta el pipeline completo de análisis de ofertas de empleo:
1. **Auth**: Autentica con Gmail OAuth 2.0
2. **Extract**: Descarga emails de ofertas desde Gmail
3. **Process**: Limpia HTML y extrae campos (empresa, puesto, salario, requisitos)
4. **Analyze**: Usa Gemini 1.5 Pro para puntuar y resumir ofertas
5. **Report**: Genera reportes en JSON y CSV

## Instrucciones
- Ejecuta cada celda en orden
- Revisa los outputs para validar cada paso
- Si hay errores, consulta la documentación en el plan del proyecto

## CELDA 1: Setup - Imports y Validación de Config

In [ ]:
# Añadir raíz del proyecto al path
import sys
import os
from pathlib import Path

# Detectar raíz del proyecto
notebook_dir = Path.cwd()
project_root = notebook_dir.parent  # Subir desde notebooks/ a raíz
sys.path.insert(0, str(project_root))

print(f"📁 Notebook dir: {notebook_dir}")
print(f"📁 Project root: {project_root}")

# Cargar y validar config
try:
    import config
    print("✅ Config cargada correctamente")
    print(f"   - GEMINI_API_KEY: {config.GEMINI_API_KEY[:20]}...")
    print(f"   - DATA_REPORTS_DIR: {config.DATA_REPORTS_DIR}")
except EnvironmentError as e:
    print(f"❌ Error en config: {e}")
    sys.exit(1)

## CELDA 2: Import de Módulos del Pipeline

In [ ]:
# Importar módulos del pipeline
try:
    from src import auth, extractor, processor, ai_analyzer, report_generator
    import pandas as pd
    print("✅ Todos los módulos importados correctamente")
except ImportError as e:
    print(f"❌ Error importando módulos: {e}")
    sys.exit(1)

## CELDA 3: Fase 1 - Autenticación Gmail OAuth 2.0

**Instrucciones:**
1. Dirígete a Google Cloud Console
2. Crea un proyecto nuevo
3. Habilita Gmail API
4. Crea credenciales OAuth 2.0 (Aplicación de escritorio)
5. Descarga el JSON y guárdalo como `credentials.json` en la raíz del proyecto
6. Ejecuta esta celda para obtener el token inicial

In [ ]:
# Ejecuta esta celda cuando tengas credentials.json en la raíz del proyecto

try:
    service = auth.get_gmail_service()
    print("✅ Autenticación Gmail exitosa")
    print(f"   Token guardado en: {config.GMAIL_TOKEN_PATH}")

    # Extracción real con query por defecto (jobalerts-noreply@linkedin.com)
    ofertas_reales = extractor.buscar_ofertas_emails(service, max_results=20)
    print(f"✅ Emails extraídos: {len(ofertas_reales)}")

except EnvironmentError as e:
    print(f"❌ {e}")
except Exception as e:
    print(f"❌ Error inesperado: {e}")

## CELDA 4: Fase 2 - Extracción de Emails (DEMO + criterio real)

Tu caso real usa el remitente `jobalerts-noreply@linkedin.com`, ya configurado como query por defecto en `src/extractor.py`.

Para esta demostración seguimos con datos simulados, pero al activar OAuth puedes ejecutar extracción real directamente.

In [ ]:
# Datos simulados para demostración
ofertas_simuladas = [
    {
        "message_id": "msg_001",
        "subject": "Job alert: Data Engineer - 5+ años Python, ETL, Spark",
        "from": "LinkedIn Job Alerts <jobalerts-noreply@linkedin.com>",
        "body_html": """
        <html>
            <body>
                <p>Nuevas ofertas para tu alerta: Data Engineer</p>
                <p><a href="https://www.linkedin.com/jobs/view/1234567890/">Ver oferta 1</a></p>
                <p><a href="https://www.linkedin.com/jobs/view/1234567891/">Ver oferta 2</a></p>
                <p>Tu alerta de empleo de LinkedIn</p>
            </body>
        </html>
        """,
        "timestamp": "2026-05-07T12:00:00"
    },
    {
        "message_id": "msg_002",
        "subject": "Job alert: Python Developer - Madrid",
        "from": "LinkedIn Job Alerts <jobalerts-noreply@linkedin.com>",
        "body_html": """
        <html>
            <body>
                <p>Hemos encontrado nuevas vacantes para Python Developer</p>
                <p><a href="https://www.linkedin.com/jobs/view/1234567892/">Ver oferta</a></p>
            </body>
        </html>
        """,
        "timestamp": "2026-05-07T11:30:00"
    }
]

print(f"📧 Ofertas simuladas: {len(ofertas_simuladas)}")
for i, oferta in enumerate(ofertas_simuladas, 1):
    print(f"   {i}. {oferta['subject'][:70]}")

## CELDA 5: Fase 3 - Procesamiento de Contenido

In [ ]:
# Procesar ofertas
print("🔧 Procesando ofertas...")
oferta_procesadas = processor.procesar_ofertas(ofertas_simuladas)

print(f"\n✅ {len(oferta_procesadas)} ofertas procesadas")

# Mostrar primera oferta procesada
if oferta_procesadas:
    print("\n📋 Ejemplo - Primera oferta procesada:")
    primera = oferta_procesadas[0]
    print(f"   Subject: {primera.get('subject', 'N/A')[:60]}")
    print(f"   Puesto: {primera.get('puesto', 'N/A')[:50]}")
    print(f"   Salario: {primera.get('salario', 'N/A')}")
    print(f"   Requisitos: {', '.join(primera.get('requisitos', []))}")

## CELDA 6: Fase 4 - Análisis con Gemini 1.5 Pro

**Importante:** Requiere GEMINI_API_KEY válida en .env

In [ ]:
# Definir perfil de usuario
perfil_usuario = """
Data Engineer con 5+ años de experiencia.
Especialidades: Python, SQL, ETL, Apache Spark, Cloud (AWS/GCP).
Buscando: Rol senior, empresa con impacto en datos, 50k+ EUR/año.
No me interesan: Roles muy junior, startups en fase seed, Latinoamérica.
"""

print("🤖 Analizando ofertas con Gemini 1.5 Pro...")
print(f"Perfil: {perfil_usuario.strip()[:80]}...")

try:
    ofertas_analizadas = ai_analyzer.analizar_lote_ofertas(oferta_procesadas, perfil_usuario)
    print(f"\n✅ Análisis completado: {len(ofertas_analizadas)} ofertas")
except Exception as e:
    print(f"❌ Error en análisis: {e}")
    print("Verifica que GEMINI_API_KEY sea válida en .env")

## CELDA 7: Fase 5 - Generación de Reportes

In [ ]:
# Generar resumen ejecutivo
try:
    report_generator.generar_resumen_ejecutivo(ofertas_analizadas)
    
    # Generar reportes en archivos
    json_path = report_generator.generar_reporte_json(ofertas_analizadas)
    csv_path = report_generator.generar_reporte_csv(ofertas_analizadas)
    
    print(f"\n✅ Reportes generados:")
    print(f"   - JSON: {json_path}")
    print(f"   - CSV: {csv_path}")
except Exception as e:
    print(f"❌ Error generando reportes: {e}")

## CELDA 8: Visualización de Datos (Pandas)

In [ ]:
# Convertir análisis a DataFrame para visualizar mejor
if ofertas_analizadas:
    df = pd.DataFrame([
        {
            'Puesto': o.get('puesto', 'N/A')[:40],
            'Empresa': o.get('empresa', 'N/A')[:25],
            'Score': o.get('score', 0),
            'Recomendación': o.get('recommendation', 'N/A'),
            'Resumen': o.get('summary', 'N/A')[:60]
        }
        for o in ofertas_analizadas
    ])
    
    print("📊 DataFrame de Ofertas:")
    print(df.to_string(index=False))
else:
    print("⚠️ No hay datos para mostrar")

## NOTAS Y PRÓXIMOS PASOS

### Setup Completo (después de pruebas)
1. **Obtener Google API Credentials**
   - Google Cloud Console → Crear proyecto
   - Habilitar: Gmail API
   - Credenciales → OAuth 2.0 (Aplicación de escritorio)
   - Descargar JSON → Renombrar a `credentials.json` en raíz

2. **Configurar .env**
   ```
   GMAIL_CLIENT_ID=xxx
   GMAIL_CLIENT_SECRET=xxx
   GEMINI_API_KEY=xxx
   ```

3. **Ejecutar Pipeline Completo**
   - Descomenta celda de autenticación
   - Ejecuta todas las celdas en orden
   - Revisa reportes en `data/reports/`

### Modelos Entrenados
- **auth.py**: Maneja OAuth 2.0, token refresh automático
- **extractor.py**: Descarga desde Gmail, maneja multipart MIME
- **processor.py**: HTML→texto con BeautifulSoup, regex para campos
- **ai_analyzer.py**: Prompting con Gemini, parsea JSON respuesta
- **report_generator.py**: Exporta JSON/CSV, resumen ejecutivo

### Trade-offs Documentados
| Decisión | Alternativa | Por qué esta | Costo |
|----------|-------------|-------------|-------|
| OAuth 2.0 | Contraseña | Seguro, estándar | Setup inicial |
| BeautifulSoup | Regex puro | Robusto | Overhead mínimo |
| Gemini API | LLM local | Inteligencia superior | Latencia + costo |
| Modular | Monolito | Testeable, escalable | Más archivos |